# Stage final — Figuras del informe (TFC)

Genera **todas las figuras numéricas** del informe `PRE-DOCUMENTO FINAL.tex` a partir de los
artefactos del repositorio (`output/optimization`, `output/evaluation`, `data/raw`), sin
reentrenar nada. Cada cifra del texto queda respaldada por una figura reproducible.

**Agrupación por tipo de comparación y horizonte** (carpetas bajo `output/figures/`):

| Grupo | Horizonte | Qué respalda |
|---|---|---|
| `dataset/` | — | Descripción del dataset: perfiles diario/semanal/anual, ACF y tendencia (líneas de la sección de dataset) |
| `optimizacion/` | — | Distribución del RMSE de la optimización, top-3, hiperparámetros y covariables vs RMSE (etapas de búsqueda) |
| `seleccion/24h`, `seleccion/3h` | por horizonte | Validación cruzada temporal: los diez candidatos y promedio vs agrupado (Tabla 10) |
| `evaluacion/24h`, `evaluacion/3h` | por horizonte | Métricas de test, RMSE vs R², percentiles del error y distribución del error (Tabla 11) |
| `residuos/24h`, `residuos/3h` | por horizonte | Diagnóstico de residuos: serie, normalidad, autocorrelación, homocedasticidad |
| `integral/24h`, `integral/3h` | por horizonte | Comparación integral: estacionalidad, fin de semana, sesgos, costo computacional |

Convenciones (idénticas a `src/analysis/residual_analysis.py` y a los YAMLs de evaluación):

- residuo = **observado − predicho** (positivo = subestimación);
- estaciones del hemisferio sur: verano = dic/ene/feb, otoño = mar/abr/may, invierno = jun/jul/ago, primavera = sep/oct/nov;
- día de semana 0 = lunes … 6 = domingo.

Al final se escribe `output/figures/README.md` con el manifiesto archivo → afirmación del informe.

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
from scipy.stats import probplot, shapiro

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")

ROOT = Path("..") if (Path("..") / "output").exists() else Path(".")
OUT = ROOT / "output" / "figures"

MODELOS = ["sarimax", "lightgbm", "lstm", "chronos2"]
NOMBRE = {"sarimax": "SARIMAX", "lightgbm": "LightGBM", "lstm": "LSTM", "chronos2": "Chronos-2"}
PALETA = dict(zip(MODELOS, ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"]))
RMSE_COL = {"sarimax": "user_attrs_val_rmse", "lightgbm": "val_rmse",
            "lstm": "val_rmse", "chronos2": "rmse_val"}
FIT_COL = {"sarimax": "user_attrs_fit_seconds", "lightgbm": "fit_seconds",
           "lstm": "fit_seconds", "chronos2": "fit_seconds"}

ESTACIONES = ["invierno", "primavera", "verano", "otoño"]
MESES_ES = ["ene", "feb", "mar", "abr", "may", "jun", "jul", "ago", "sep", "oct", "nov", "dic"]
DOW_ES = ["lunes", "martes", "miércoles", "jueves", "viernes", "sábado", "domingo"]

from decimal import Decimal, ROUND_DOWN

MANIFIESTO = []


def trunc4(x):
    """Trunca a 4 decimales (sin redondear), como en el informe."""
    return float(Decimal(str(float(x))).quantize(Decimal("0.0001"),
                                                rounding=ROUND_DOWN))


def fmt4(x, signo=False):
    """Cuatro decimales truncados con coma decimal (convencion del informe)."""
    q = Decimal(str(float(x))).quantize(Decimal("0.0001"), rounding=ROUND_DOWN)
    return (f"{q:+.4f}" if signo else f"{q:.4f}").replace(".", ",")


def guardar(fig, grupo, nombre, horizonte=None, afirmacion=""):
    """Guarda la figura en output/figures/<grupo>[/<h>h]/ y la registra en el manifiesto."""
    carpeta = OUT / grupo if horizonte is None else OUT / grupo / f"{horizonte}h"
    carpeta.mkdir(parents=True, exist_ok=True)
    base = f"{grupo}-{horizonte}h-{nombre}.png" if horizonte else f"{grupo}-{nombre}.png"
    ruta = carpeta / base
    fig.savefig(ruta, dpi=150, bbox_inches="tight")
    plt.close(fig)
    MANIFIESTO.append({"archivo": str(ruta.relative_to(OUT)).replace("\\", "/"),
                       "grupo": grupo, "horizonte": horizonte, "afirmacion": afirmacion})
    print(f"  guardada -> {ruta.relative_to(ROOT)}")


print(f"ROOT = {ROOT.resolve()}")
print(f"OUT  = {OUT.resolve()}")

## 0. Carga de artefactos

In [ ]:
trials, n_total = {}, {}
for m in MODELOS:
    d = pd.read_csv(ROOT / "output" / "optimization" / f"{m}_trials.csv")
    n_total[m] = len(d)
    if "state" in d.columns:
        d = d[d["state"].astype(str).str.contains("COMPLETE")]
    d = d.copy()
    d["rmse"] = pd.to_numeric(d[RMSE_COL[m]], errors="coerce")
    d = d[np.isfinite(d["rmse"])]
    d["trial"] = d["trial_number"] if "trial_number" in d.columns else d["number"]
    trials[m] = d
    print(f"{NOMBRE[m]:10s}: {len(d):3d} trials con RMSE finito (registrados: {n_total[m]})")

cv = {(m, h): pd.read_csv(ROOT / "output" / "evaluation" / f"{m}_cv_top10_{h}h.csv")
      for m in MODELOS for h in (24, 3)}
ganadores = {(m, h): yaml.safe_load(open(ROOT / "output" / "evaluation" / f"{m}_cv_winner_{h}h.yaml",
                                       encoding="utf-8"))
             for m in MODELOS for h in (24, 3)}
pred = {(m, h): pd.read_csv(ROOT / "output" / "evaluation" / f"{m}_test_predictions_{h}h.csv",
                            parse_dates=["datetime"])
        for m in MODELOS for h in (24, 3)}
resid = {(m, h): yaml.safe_load(open(ROOT / "output" / "evaluation" / "residuals" /
                                     f"{m}_{h}h_residual_analysis.yaml", encoding="utf-8"))
         for m in MODELOS for h in (24, 3)}
print("\nCarga completa: trials, candidatos de validación cruzada, ganadores, predicciones y residuos.")

## 1. Dataset (sin horizonte) — perfiles, autocorrelación y tendencia

In [ ]:
df = pd.read_csv(ROOT / "data" / "raw" / "demandas_horarias_2023_2026.csv", encoding="utf-8")
df.columns = (df.columns.astype(str).str.replace("\n", " ", regex=False)
              .str.replace(r"\s+", " ", regex=True).str.strip())
df = df[["FECHA", "Horario", "TOTAL_SIP"]].copy()
df["FECHA"] = pd.to_datetime(df["FECHA"], errors="coerce")
df["Horario"] = df["Horario"].astype(str).str.strip()
df = df[df["Horario"].str.match(r"^\d{1,2}:00(:00)?$")].copy()
df["TOTAL_SIP"] = pd.to_numeric(df["TOTAL_SIP"], errors="coerce")
df = df[df["TOTAL_SIP"].notna() & (df["TOTAL_SIP"] > 0)].copy()
df["datetime"] = pd.to_datetime(df["FECHA"].dt.strftime("%Y-%m-%d") + " " +
                                df["Horario"].str.replace(" ", ""), errors="coerce")
df = df[df["datetime"].notna()].copy()
df = (df.sort_values("datetime").drop_duplicates(subset="datetime")
      .set_index("datetime")[["TOTAL_SIP"]].rename(columns={"TOTAL_SIP": "demanda"}))
df = df.reindex(pd.date_range(df.index.min(), df.index.max(), freq="h"))
df["demanda"] = df["demanda"].interpolate(method="time")
df = df.dropna(subset="demanda").loc[: "2026-01-01 00:00:00"]

df["hora"] = df.index.hour
df["mes"] = df.index.month
df["dow"] = df.index.dayofweek
df["estacion"] = df["mes"].map({12: "verano", 1: "verano", 2: "verano",
                                3: "otoño", 4: "otoño", 5: "otoño",
                                6: "invierno", 7: "invierno", 8: "invierno",
                                9: "primavera", 10: "primavera", 11: "primavera"})
print(f"serie modelada: {len(df)} horas ({df.index.min()} -> {df.index.max()})")

In [ ]:
# Perfil diario: invierno vs verano (respalda la descripcion del pico al mediodia/noche)
perfil = df.groupby(["hora", "estacion"])["demanda"].mean().unstack()
fig, ax = plt.subplots(figsize=(9, 4.5))
for est, color in [("invierno", "#1f77b4"), ("verano", "#d62728")]:
    ax.plot(perfil.index, perfil[est], lw=2, color=color, label=est.capitalize())
ax.set_xlabel("Hora del día")
ax.set_ylabel("Demanda promedio (MW)")
ax.set_title("Perfil diario promedio: invierno vs verano")
ax.set_xticks(range(0, 24, 2))
ax.legend()
guardar(fig, "dataset", "perfil-diario",
        afirmacion="Diferencia de perfiles diarios: pico al mediodía en verano y de noche en invierno.")

In [ ]:
# Perfil semanal: descenso marcado los domingos (respalda la descripcion semanal)
semanal = df.groupby("dow")["demanda"].mean()
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(range(7), [semanal.get(i, np.nan) for i in range(7)],
       color=["#4c78a8" if i not in (0, 6) else "#d62728" for i in range(7)])
ax.set_xticks(range(7))
ax.set_xticklabels(DOW_ES, rotation=30)
ax.set_ylabel("Demanda promedio (MW)")
ax.set_title("Perfil semanal promedio (domingo y lunes destacados)")
guardar(fig, "dataset", "perfil-semanal",
        afirmacion="Descenso marcado del consumo los domingos; dias laborables superiores.")

In [ ]:
# Perfil anual promedio por mes (respalda la descripcion de estacionalidad anual)
anual = df.groupby("mes")["demanda"].mean()
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(range(1, 13), anual.reindex(range(1, 13)), marker="o", color="#2ca02c")
ax.set_xticks(range(1, 13))
ax.set_xticklabels(MESES_ES)
ax.set_ylabel("Demanda promedio (MW)")
ax.set_title("Perfil anual promedio por mes")
guardar(fig, "dataset", "perfil-anual",
        afirmacion="Estacionalidad anual de la demanda (meses de verano vs invierno).")

In [ ]:
# ACF/PACF de la demanda: ciclos diario y semanal (respalda la figura de autocorrelacion)
from statsmodels.tsa.stattools import acf, pacf

x = df["demanda"].values
ac = acf(x, nlags=168)
pc = pacf(x, nlags=48)
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
axes[0].bar(range(len(ac)), ac, width=1.0, color="#4c78a8")
axes[0].axvline(24, color="red", ls="--", lw=1, label="24 h")
axes[0].axvline(168, color="darkred", ls="--", lw=1, label="168 h")
axes[0].set_title("ACF de la demanda (168 rezagos)")
axes[1].bar(range(len(pc)), pc, width=1.0, color="#4c78a8")
axes[1].axvline(24, color="red", ls="--", lw=1)
axes[1].set_title("PACF de la demanda (48 rezagos)")
for ax in axes:
    ax.set_xlabel("Rezago (horas)")
    ax.legend()
fig.tight_layout()
guardar(fig, "dataset", "autocorrelacion",
        afirmacion="Ciclos diario (24 h) y semanal (168 h) de la demanda; justifica m=24 y ventana de contexto 168 h.")

In [ ]:
# Tendencia: MSTL + Theil-Sen robusto (respalda la pendiente de 8,32 MW/anio)
from statsmodels.tsa.seasonal import MSTL
from sklearn.linear_model import TheilSenRegressor

cache = OUT / "dataset" / "tendencia_mstl_cache.csv"
cache.parent.mkdir(parents=True, exist_ok=True)
if cache.exists():
    # float_precision="round_trip" es imprescindible: el parser por defecto
    # introduce desvios de un ulp que cambian la mediana de Theil-Sen (8,32 -> 8,55)
    tendencia = pd.read_csv(cache, index_col=0, float_precision="round_trip")
    tendencia.index = pd.to_datetime(tendencia.index)
    tendencia = tendencia["tendencia"]
    print("tendencia MSTL cargada del cache")
else:
    print("calculando MSTL (periodos 24, 168, 8760)... puede tardar varios minutos")
    tendencia = MSTL(df["demanda"], periods=(24, 168, 8760)).fit().trend.dropna()
    tendencia.rename("tendencia").to_csv(cache, index_label="datetime")

X = np.arange(len(tendencia)).reshape(-1, 1)
recta = TheilSenRegressor(random_state=0).fit(X, tendencia.values)
pend_anual = recta.coef_[0] * 24 * 365.25

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(tendencia.index, df["demanda"].rolling(168, center=True).mean().reindex(tendencia.index),
        lw=1, color="#999999", label="Demanda (media 168 h)")
ax.plot(tendencia.index, recta.predict(X), lw=2.5, color="#d62728",
        label=f"Tendencia Theil-Sen: {pend_anual:.2f} MW/año")
ax.set_ylabel("Demanda (MW)")
ax.set_title("Tendencia estructural de la demanda (MSTL + Theil-Sen)")
ax.legend()
guardar(fig, "dataset", "tendencia",
        afirmacion=f"Pendiente anual equivalente de apenas {pend_anual:.2f} MW/año: sin tendencia significativa.")
print(f"\npendiente anual equivalente = {pend_anual:.2f} MW/año (informe: 8,32)")

## 2. Optimización de hiperparámetros y selección de variables (sin horizonte)

In [ ]:
# Distribucion del RMSE de la optimizacion + top-3 (figuras del informe)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
for m in MODELOS:
    v = trials[m]["rmse"]
    axes[0].violinplot([v.values], positions=[MODELOS.index(m)], widths=0.9,
                       showextrema=False)
    axes[0].scatter(MODELOS.index(m), v.min(), color="red", s=40, zorder=5)
top3 = {m: trials[m].nsmallest(3, "rmse") for m in MODELOS}
ancho = 0.25
for k in range(3):
    pos = np.arange(len(MODELOS)) + (k - 1) * ancho
    vals = [top3[m]["rmse"].iloc[k] for m in MODELOS]
    axes[1].bar(pos, vals, width=ancho, color=sns.color_palette("Blues", 5)[4 - k],
                edgecolor="navy", label=f"Top {k + 1}")
    for p, val in zip(pos, vals):
        axes[1].text(p, val, f"{val:.2f}", ha="center", va="bottom", fontsize=8)
axes[0].set_xticks(range(len(MODELOS)))
axes[0].set_xticklabels([NOMBRE[m] for m in MODELOS])
axes[0].set_ylabel("RMSE de la optimización (MW)")
axes[0].set_title("Distribución del RMSE de la optimización")
axes[1].set_xticks(np.arange(len(MODELOS)))
axes[1].set_xticklabels([NOMBRE[m] for m in MODELOS])
axes[1].set_ylim(20, None)
axes[1].set_ylabel("RMSE de la optimización (MW)")
axes[1].set_title("Tres mejores trials por modelo")
axes[1].legend()
fig.tight_layout()
guardar(fig, "optimizacion", "distribucion-y-top3",
        afirmacion="Distribuciones por modelo y diferencias minúsculas entre los tres mejores (0,15 / 0,06 / 0,03 / 0).")

In [ ]:
# Hiperparametros vs RMSE (con r de Pearson): respalda las correlaciones del texto
HIPER = {
    "sarimax": ["params_P", "params_Q", "params_p"],
    "lightgbm": ["learning_rate", "max_depth", "lambda_l2", "min_child_samples", "n_estimators"],
    "lstm": ["hidden_size", "dropout", "num_layers", "epochs"],
    "chronos2": [],
}
fig, axes = plt.subplots(len(MODELOS), 5, figsize=(20, 12))
for i, m in enumerate(MODELOS):
    d = trials[m]
    for j in range(5):
        ax = axes[i, j]
        if j < len(HIPER[m]):
            col = HIPER[m][j]
            r = d[col].corr(d["rmse"])
            ax.scatter(d[col], d["rmse"], s=12, alpha=0.6, color=PALETA[m])
            ax.set_xlabel(col)
            ax.text(0.03, 0.95, f"r = {fmt4(r, signo=True)}", transform=ax.transAxes,
                    va="top", fontsize=10, bbox=dict(fc="white", ec="gray", alpha=0.8))
        else:
            ax.axis("off")
        if j == 0:
            ax.set_ylabel(f"{NOMBRE[m]}\nRMSE (MW)")
axes[0, 0].set_title("Hiperparámetros vs RMSE de la optimización (Pearson)", loc="left")
fig.tight_layout()
guardar(fig, "optimizacion", "hiperparametros-vs-rmse",
        afirmacion="Correlaciones de Pearson entre hiperparámetros y el RMSE de la optimización por modelo.")

In [ ]:
# Covariables vs RMSE: r de Pearson de la inclusion binaria (o del orden, para Fourier)
def r_inclusion(d, col):
    return d[col].astype(int).corr(d["rmse"])

COV = {}
s = trials["sarimax"]
COV["sarimax"] = {c.replace("params_use_", ""): r_inclusion(s, c) for c in
                  ["params_use_target_lags", "params_use_HUM", "params_use_TEMP",
                   "params_use_dia_semana", "params_use_dia_anio", "params_use_mes",
                   "params_use_es_laboral"]}
COV["sarimax"]["fourier_k_weekly>=3"] = (s["params_k_weekly"] >= 3).astype(int).corr(s["rmse"])
l = trials["lightgbm"]
COV["lightgbm"] = {c.replace("use_", ""): r_inclusion(l, c) for c in
                   ["use_temp_future", "use_hum_lags", "use_indice_temporal", "use_temp_lags",
                    "use_feriado"]}
COV["lightgbm"]["fourier_k_weekly"] = l["fourier_k_weekly"].corr(l["rmse"])
COV["lightgbm"]["fourier_k_yearly"] = l["fourier_k_yearly"].corr(l["rmse"])
t = trials["lstm"]
COV["lstm"] = {c.replace("use_", ""): r_inclusion(t, c) for c in
               ["use_feriado", "use_humedad", "use_dia_anio", "use_temperatura"]}
c = trials["chronos2"]
COV["chronos2"] = {nombre: c["selected_future_cols"].astype(str).str.contains(nombre)
                   .astype(int).corr(c["rmse"])
                   for nombre in ["es_laboral", "TEMP_F_24", "HUM_F_24", "hum_lag_24h",
                                  "temp_lag_24h", "target_lag_24"]}

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for ax, m in zip(axes.ravel(), MODELOS):
    r = pd.Series(COV[m]).dropna().sort_values()
    ax.barh(range(len(r)), r.values, color=[PALETA[m] if v > 0 else "#d62728" for v in r.values])
    ax.set_yticks(range(len(r)))
    ax.set_yticklabels(r.index, fontsize=8)
    for k, v in enumerate(r.values):
        ax.text(v, k, f" {fmt4(v, signo=True)}", va="center", fontsize=8,
                ha="left" if v >= 0 else "right")
    ax.axvline(0, color="black", lw=0.8)
    ax.set_xlim(min(r.values.min() - 0.2, -1), max(r.values.max() + 0.2, 1))
    ax.set_title(f"{NOMBRE[m]}: r de la covariable vs RMSE")
fig.suptitle("Covariables candidatas vs RMSE de la optimización", y=1.01)
fig.tight_layout()
guardar(fig, "optimizacion", "covariables-vs-rmse",
        afirmacion="Correlación entre la inclusión de cada covariable candidata y el RMSE de la optimización.")

## 3. Selección mediante validación cruzada temporal — por horizonte

In [ ]:
# Los diez candidatos de cada modelo, con el optimo marcado
for h in (24, 3):
    fig, axes = plt.subplots(2, 2, figsize=(13, 7.5))
    for ax, m in zip(axes.ravel(), MODELOS):
        d = cv[(m, h)].sort_values("cv_rmse").reset_index(drop=True)
        colores = [PALETA[m]] * len(d)
        gan = int(ganadores[(m, h)]["trial_number"])
        d["es_ganador"] = d["trial_number"] == gan
        colores = ["#d62728" if e else PALETA[m] for e in d["es_ganador"]]
        ax.bar(range(len(d)), d["cv_rmse"], color=colores)
        for k, row in d.iterrows():
            ax.text(k, row["cv_rmse"], f"{int(row['trial_number'])}", ha="center",
                    va="bottom", fontsize=7, rotation=90)
        ax.set_xticks([])
        ax.set_ylabel("RMSE de selección (MW)")
        vmin, vmax = d["cv_rmse"].min(), d["cv_rmse"].max()
        ax.set_ylim(vmin - 0.05 * (vmax - vmin + 1), vmax + 0.12 * (vmax - vmin + 1))
        ax.set_title(f"{NOMBRE[m]} — óptimo: trial {gan} ({d['cv_rmse'].min():.2f} MW)")
    fig.suptitle(f"RMSE de selección de los diez candidatos (horizonte {h} h); números = trial", y=1.0)
    fig.tight_layout()
    guardar(fig, "seleccion", "candidatos", horizonte=h,
            afirmacion=f"Orden de los modelos y rangos de candidatos a {h} h; SARIMAX colapsa con el trial 80.")

In [ ]:
# Promedio pliegue a pliegue vs RMSE agrupado (respalda el orden estable)
for h in (24, 3):
    fig, ax = plt.subplots(figsize=(8, 4.2))
    ancho = 0.38
    resumen = pd.DataFrame({
        "promedio": {m: cv[(m, h)]["cv_rmse"].min() for m in MODELOS},
        "agrupado": {m: cv[(m, h)]["cv_rmse_pooled"].min() for m in MODELOS},
    })
    resumen.plot.bar(ax=ax, width=ancho, color=["#4c78a8", "#f58518"])
    for i, m in enumerate(MODELOS):
        for j, col in enumerate(resumen.columns):
            ax.text(i + (j - 0.5) * ancho, resumen.loc[m, col], f"{resumen.loc[m, col]:.2f}",
                    ha="center", va="bottom", fontsize=8)
    ax.set_xticklabels([NOMBRE[m] for m in MODELOS], rotation=0)
    ax.set_ylabel("RMSE de selección (MW)")
    ax.set_ylim(0, resumen.values.max() * 1.15)
    ax.set_title(f"Promedio pliegue a pliegue vs RMSE agrupado ({h} h): mismo orden")
    ax.legend(["Promedio de los cuatro pliegues", "Agrupado (predicciones concatenadas)"])
    fig.tight_layout()
    guardar(fig, "seleccion", "promedio-vs-agrupado", horizonte=h,
            afirmacion=f"El orden por RMSE agregado es idéntico al promedio a {h} h.")

## 4. Evaluación sobre el conjunto de test — por horizonte

In [ ]:
# Metricas de test (MAE, RMSE, MAPE, SMAPE, R2) por horizonte (Tabla 11)
METRICAS = [("mae", "MAE (MW)"), ("rmse", "RMSE (MW)"), ("mape", "MAPE (%)"),
            ("smape", "SMAPE (%)"), ("r2", "R²")]
for h in (24, 3):
    tm = pd.DataFrame({NOMBRE[m]: ganadores[(m, h)]["test_metrics"] for m in MODELOS}).T
    fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))
    for ax, cols, titulo in [(axes[0], ["mae", "rmse"], "Errores absolutos (MW)"),
                             (axes[1], ["mape", "smape"], "Errores relativos (%)"),
                             (axes[2], ["r2"], "Coeficiente de determinación")]:
        tm[cols].plot.bar(ax=ax, color=["#4c78a8", "#f58518"][:len(cols)], rot=0)
        for cont in ax.containers:
            ax.bar_label(cont, labels=[fmt4(v) for v in cont.datavalues],
                     fontsize=8, padding=2)
        ax.set_title(titulo)
        ax.set_xlabel("")
        ax.set_ylim(0, 1.0) if cols == ["r2"] else None
    fig.suptitle(f"Métricas de evaluación sobre test (horizonte {h} h)", y=1.02)
    fig.tight_layout()
    guardar(fig, "evaluacion", "metricas-test", horizonte=h,
            afirmacion=f"Valores de la Tabla 11 y del párrafo de evaluación a {h} h.")

In [ ]:
# RMSE vs R2 con etiqueta de modelo y del ganador de la seleccion (figura del informe)
fig, axes = plt.subplots(1, 2, figsize=(13, 5.2))
for ax, h in zip(axes, (24, 3)):
    for m in MODELOS:
        tm = ganadores[(m, h)]["test_metrics"]
        mejor = m == min(MODELOS, key=lambda x: ganadores[(x, h)]["test_metrics"]["rmse"])
        ax.scatter(tm["rmse"], tm["r2"], s=140, color=PALETA[m],
                   edgecolor="navy" if mejor else "black", linewidth=2 if mejor else 0.8, zorder=3)
        ax.annotate(f"{NOMBRE[m]}\ntrial {ganadores[(m, h)]['trial_number']}",
                    (tm["rmse"], tm["r2"]), textcoords="offset points", xytext=(8, 6), fontsize=9)
    ax.set_xlabel("RMSE de test (MW)")
    ax.set_ylabel("R²")
    ax.set_title(f"Horizonte {h} h; contorno = mejor modelo")
fig.suptitle("RMSE vs R² sobre el conjunto de test", y=1.0)
fig.tight_layout()
guardar(fig, "evaluacion", "rmse-vs-r2",
        afirmacion="Posicionamiento de los cuatro modelos por RMSE y R² sobre test en los dos horizontes; contorno = mejor modelo de cada horizonte.")

In [ ]:
# Desempeno final: barras agrupadas de RMSE y R2 por modelo y horizonte (figura del informe)
datos = {clave: pd.DataFrame(
    {h: {NOMBRE[m]: ganadores[(m, h)]["test_metrics"][clave] for m in MODELOS} for h in (24, 3)})
    for clave in ("rmse", "r2")}
ancho = 0.38
fig, axes = plt.subplots(1, 2, figsize=(13, 5.2))
for ax, clave, titulo, ylabel, leyenda in [
        (axes[0], "rmse", "RMSE por horizonte", "RMSE (MW)", "upper right"),
        (axes[1], "r2", "R² por horizonte", "R²", "upper left")]:
    df = datos[clave]
    df.plot.bar(ax=ax, width=ancho, color=["#4c78a8", "#f58518"], rot=0)
    for i, nombre in enumerate(df.index):
        for j, h in enumerate((24, 3)):
            v = df.loc[nombre, h]
            ax.text(i + (j - 0.5) * ancho, v, fmt4(v),
                    ha="center", va="bottom", fontsize=8)
    ax.set_xticklabels(df.index, rotation=0)
    ax.set_ylabel(ylabel)
    ax.set_xlabel("")
    ax.set_title(titulo)
    ax.set_ylim(0, 1.0 if clave == "r2" else df.values.max() * 1.15)
    ax.legend(["Day-Ahead (24 h)", "Hour-Ahead (3 h)"],
              title="Horizonte de predicción", loc=leyenda, fontsize=9, title_fontsize=9)
fig.suptitle("Desempeño final sobre el conjunto de test, por modelo y horizonte", y=1.0)
fig.tight_layout()
guardar(fig, "evaluacion", "desempeno-final",
        afirmacion="Desempeño de los cuatro modelos sobre test en ambos horizontes, medido con RMSE y R².")

In [ ]:
# Percentiles del error absoluto (p50/p90/p95) sobre test, ambos horizontes
pcts = {}
for h in (24, 3):
    ae = pd.DataFrame({NOMBRE[m]: (pred[(m, h)]["predicted"] - pred[(m, h)]["actual"]).abs()
                       for m in MODELOS})
    pcts[h] = ae.quantile([0.5, 0.9, 0.95]).T

fig, axes = plt.subplots(1, 2, figsize=(16, 4.6))
for ax, h in zip(axes, (24, 3)):
    pcts[h].plot.bar(ax=ax, color=["#4c78a8", "#d62728", "#72b7b2"], rot=0)
    for cont in ax.containers:
        ax.bar_label(cont, labels=[fmt4(v) for v in cont.datavalues], fontsize=8, padding=2)
    ax.set_title(f"Percentiles del error absoluto (MW) \u2013 Colas del error sobre test ({h} h)")
    ax.set_xlabel("")
    ax.set_ylabel("|error| (MW)")
    ax.legend(title="Percentil", fontsize=8, title_fontsize=8)
fig.tight_layout()
guardar(fig, "evaluacion", "percentiles-error", horizonte=None,
        afirmacion="Percentiles p50/p90/p95 del error absoluto sobre test en ambos horizontes.")
print("p90 24h: " + ", ".join(f"{NOMBRE[m]}={pcts[24].loc[NOMBRE[m], 0.9]:.4f}" for m in MODELOS))
print("p90 3h:  " + ", ".join(f"{NOMBRE[m]}={pcts[3].loc[NOMBRE[m], 0.9]:.4f}" for m in MODELOS))


## 5. Análisis de residuos — por horizonte

In [ ]:
# Residuos = observado - predicho (misma convencion que src/analysis/residual_analysis.py)
residuos = {}
for m in MODELOS:
    for h in (24, 3):
        d = pred[(m, h)].copy()
        d["resid"] = d["actual"] - d["predicted"]
        residuos[(m, h)] = d

for h in (24, 3):
    fig, axes = plt.subplots(4, 1, figsize=(12, 10), sharex=True)
    for ax, m in zip(axes, MODELOS):
        d = residuos[(m, h)]
        ax.plot(d["datetime"], d["resid"], lw=0.5, color=PALETA[m])
        ax.axhline(0, color="black", lw=0.8)
        media = resid[(m, h)]["sesgos_sistematicos"]["mean_bias"]
        ax.set_ylabel(f"{NOMBRE[m]}\n(MW)")
        ax.text(0.005, 0.85, f"sesgo medio = {fmt4(media, signo=True)} MW", transform=ax.transAxes,
                fontsize=8, va="top", bbox=dict(fc="white", ec="gray", alpha=0.7))
    axes[0].set_title(f"Serie de residuos sobre test (horizonte {h} h)")
    fig.tight_layout()
    guardar(fig, "residuos", "serie", horizonte=h,
            afirmacion=f"Serie de residuos a {h} h y sesgo medio de cada modelo.")

In [ ]:
# Normalidad: histograma + QQ, con Shapiro y curtosis del YAML (linea del informe)
for h in (24, 3):
    fig, axes = plt.subplots(len(MODELOS), 2, figsize=(11, 11))
    for i, m in enumerate(MODELOS):
        r = residuos[(m, h)]["resid"].values
        y = resid[(m, h)]
        axes[i, 0].hist(r, bins=60, density=True, color=PALETA[m], alpha=0.75)
        axes[i, 0].axvline(0, color="black", lw=0.8)
        axes[i, 0].set_ylabel(NOMBRE[m])
        probplot(r, dist="norm", plot=axes[i, 1])
        axes[i, 1].set_title("")
        axes[i, 0].text(0.62, 0.85,
                        f"Shapiro = {fmt4(y['normalidad']['shapiro_stat'])}\n"
                        f"curtosis = {fmt4(y['tamano_efecto']['excess_kurtosis'])}\n"
                        f">3σ = {fmt4(y['tamano_efecto']['pct_beyond_3sigma'])} %",
                        transform=axes[i, 0].transAxes, fontsize=8, va="top",
                        bbox=dict(fc="white", ec="gray", alpha=0.7))
    axes[0, 0].set_title("Distribución de residuos")
    axes[0, 1].set_title("QQ-plot normal")
    fig.suptitle(f"Normalidad de los residuos ({h} h)", y=1.0)
    fig.tight_layout()
    guardar(fig, "residuos", "normalidad", horizonte=h,
            afirmacion=f"Rechazo de normalidad, asimetría y colas pesadas a {h} h (Shapiro, curtosis, % >3σ).")

In [ ]:
# Autocorrelacion: ACF y PACF de los residuos por modelo
from statsmodels.tsa.stattools import acf as _acf, pacf as _pacf

for h in (24, 3):
    fig, axes = plt.subplots(len(MODELS := MODELOS), 2, figsize=(12, 11))
    for i, m in enumerate(MODELOS):
        r = residuos[(m, h)]["resid"].values
        y = resid[(m, h)]
        ac = _acf(r, nlags=168)
        pc = _pacf(r, nlags=48)
        axes[i, 0].bar(range(len(ac)), ac, width=1.0, color=PALETA[m])
        axes[i, 1].bar(range(len(pc)), pc, width=1.0, color=PALETA[m])
        axes[i, 0].set_ylabel(f"{NOMBRE[m]}\nACF")
        axes[i, 1].set_ylabel("PACF")
        axes[i, 0].text(0.72, 0.85,
                        f"ACF(1) = {fmt4(y['autocorrelacion']['acf_lag_1'])}\n"
                        f"var. explicada = {fmt4(y['autocorrelacion']['acf1_variance_explained_pct'])} %",
                        transform=axes[i, 0].transAxes, fontsize=8, va="top",
                        bbox=dict(fc="white", ec="gray", alpha=0.7))
    axes[0, 0].set_title("ACF (168 rezagos)")
    axes[0, 1].set_title("PACF (48 rezagos)")
    axes[-1, 0].set_xlabel("Rezago (horas)")
    axes[-1, 1].set_xlabel("Rezago (horas)")
    fig.suptitle(f"Autocorrelación de los residuos ({h} h)", y=1.0)
    fig.tight_layout()
    guardar(fig, "residuos", "autocorrelacion", horizonte=h,
            afirmacion=f"Estructura residual: ACF(1) entre 0,39 y 0,82 y varianza explicada a {h} h.")

In [ ]:
# Homocedasticidad: |residuos| vs predichos con Spearman y Goldfeld-Quandt del YAML
for h in (24, 3):
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    for ax, m in zip(axes.ravel(), MODELOS):
        d = residuos[(m, h)]
        y = resid[(m, h)]["homocedasticidad"]
        ax.scatter(d["predicted"], d["resid"].abs(), s=6, alpha=0.4, color=PALETA[m])
        ax.axhline(0, color="black", lw=0.8)
        ax.set_xlabel("Predicho (MW)")
        ax.set_ylabel("|residuo| (MW)")
        ax.set_title(f"{NOMBRE[m]} — Spearman = {fmt4(y['spearman_corr'])}, "
                     f"GQ(por nivel) p = {fmt4(y['goldfeld_quandt_by_prediction_pval'])}")
    fig.suptitle(f"Constancia de la varianza ({h} h)", y=1.0)
    fig.tight_layout()
    guardar(fig, "residuos", "homocedasticidad", horizonte=h,
            afirmacion=f"Correlación de Spearman y pruebas de Goldfeld-Quandt a {h} h.")

## 6. Comparación integral — por horizonte (estacionalidad, calendario y sesgos)

In [ ]:
# MAE por estacion (barras agrupadas por estacion, un panel por modelo)
for h in (24, 3):
    fig, axes = plt.subplots(2, 2, figsize=(12, 7.5), sharey=False)
    for ax, m in zip(axes.ravel(), MODELOS):
        y = resid[(m, h)]["sesgos_sistematicos"]["mae_by_season"]
        serie = pd.Series(y).reindex([e for e in ESTACIONES if e in y])
        serie.plot.bar(ax=ax, color=PALETA[m], rot=0)
        for cont in ax.containers:
            ax.bar_label(cont, labels=[fmt4(v) for v in cont.datavalues], fontsize=8, padding=2)
        ax.set_title(f"{NOMBRE[m]} — verano/invierno = {fmt4(serie.get('verano', np.nan) / serie.get('invierno', np.nan))}×")
        ax.set_xlabel("")
        ax.set_ylabel("MAE (MW)")
    fig.suptitle(f"MAE de test por estación ({h} h)", y=1.0)
    fig.tight_layout()
    guardar(fig, "integral", "mae-por-estacion", horizonte=h,
            afirmacion=f"MAE estacional y ratio verano/invierno de cada modelo a {h} h.")
    print(f"{h}h estaciones: " +
          " | ".join(f"{NOMBRE[m]}: " +
                     ", ".join(f"{k}={v:.2f}" for k, v in resid[(m, h)]["sesgos_sistematicos"]["mae_by_season"].items())
                     for m in MODELOS))

In [ ]:
# Efecto de fin de semana: MAE del finde (sábado y domingo) vs MAE laboral, y MAE por dia
penal_h, maes_dow_h = {}, {}
for h in (24, 3):
    penal, maes_dow = {}, {}
    for m in MODELOS:
        d = pred[(m, h)]
        e = (d["predicted"] - d["actual"]).abs()
        dow = d["datetime"].dt.dayofweek
        penal[m] = (e[dow >= 5].mean() / e[dow <= 4].mean() - 1) * 100
        maes_dow[m] = e.groupby(dow).mean().reindex(range(7))
    penal_h[h], maes_dow_h[h] = penal, maes_dow

fig = plt.figure(figsize=(14, 9))
subs = fig.subfigures(2, 1, hspace=0.06)
for sub, h in zip(subs, (24, 3)):
    sub.suptitle(f"Efecto de fin de semana ({h} h)", fontsize=13)
    axs = sub.subplots(1, 2)
    penal = pd.Series(penal_h[h]).sort_values()
    axs[0].barh(range(len(penal)), penal.values, color=[PALETA[m] for m in penal.index])
    axs[0].set_yticks(range(len(penal)))
    axs[0].set_yticklabels([NOMBRE[m] for m in penal.index])
    for k, v in enumerate(penal.values):
        axs[0].text(v, k, f" {fmt4(v)} %", va="center", fontsize=9)
    axs[0].set_xlim(0, max(penal.values) * 1.3)
    axs[0].set_xlabel("Penalización del fin de semana (MAE finde / MAE laboral - 1, en %)")
    axs[0].set_title("Penalización (sábado y domingo frente a lunes-viernes)")
    for m in MODELOS:
        axs[1].plot(range(7), maes_dow_h[h][m].values, marker="o", lw=1.6,
                    color=PALETA[m], label=NOMBRE[m])
    axs[1].set_xticks(range(7))
    axs[1].set_xticklabels(DOW_ES, rotation=30)
    axs[1].set_ylabel("MAE (MW)")
    axs[1].set_title("MAE por día de la semana")
    axs[1].legend(fontsize=8)
guardar(fig, "integral", "efecto-fin-de-semana", horizonte=None,
        afirmacion="Penalización de fin de semana y MAE por día, en ambos horizontes.")
print("24h penalizacion finde: " + ", ".join(f"{NOMBRE[m]}={penal_h[24][m]:.4f}%" for m in MODELOS))
print("3h penalizacion finde:  " + ", ".join(f"{NOMBRE[m]}={penal_h[3][m]:.4f}%" for m in MODELOS))


In [ ]:
# Sesgo medio por modelo y distribución del MAE por hora, ambos horizontes
fig = plt.figure(figsize=(14, 9))
subs = fig.subfigures(2, 1, hspace=0.06)
for sub, h in zip(subs, (24, 3)):
    sub.suptitle(f"Sesgos sistemáticos ({h} h)", fontsize=13)
    axs = sub.subplots(1, 2)
    sesgos = {NOMBRE[m]: resid[(m, h)]["sesgos_sistematicos"]["mean_bias"] for m in MODELOS}
    serie = pd.Series(sesgos)
    axs[0].bar(range(len(serie)), serie.values,
               color=["#2ca02c" if v > 0 else "#d62728" for v in serie.values])
    axs[0].set_xticks(range(len(serie)))
    axs[0].set_xticklabels(serie.index)
    for k, v in enumerate(serie.values):
        axs[0].text(k, v, fmt4(v, signo=True), ha="center",
                    va="bottom" if v >= 0 else "top", fontsize=9)
    axs[0].axhline(0, color="black", lw=0.8)
    axs[0].set_ylabel("Residuo medio (MW)")
    axs[0].set_title("Sesgo medio (+ subestima, − sobreestima)")
    for m in MODELOS:
        por_hora = pd.Series(resid[(m, h)]["sesgos_sistematicos"]["mae_by_hour"])
        axs[1].plot(por_hora.index, por_hora.values, lw=1.6, color=PALETA[m], label=NOMBRE[m])
    axs[1].set_xticks(range(0, 24, 2))
    axs[1].set_xlabel("Hora del día")
    axs[1].set_ylabel("MAE (MW)")
    axs[1].set_title("MAE por hora del día")
    axs[1].legend(fontsize=8)
guardar(fig, "integral", "sesgos", horizonte=None,
        afirmacion="Sesgo medio por modelo y MAE por hora del día, en ambos horizontes.")


In [ ]:
# Costo de la busqueda vs error de test; media y mediana del ajuste por trial
# SARIMAX: el costo del informe es la suma de la columna `duration` (todas las filas);
# los demas modelos suman ajuste + walk-forward, como en la Tabla 11.
costo, medias, medianas, wf_stats = {}, {}, {}, {}
for m in MODELOS:
    t = pd.read_csv(ROOT / "output" / "optimization" / f"{m}_trials.csv")
    fit = pd.to_numeric(t[FIT_COL[m]], errors="coerce")
    wf_col = "user_attrs_walkforward_seconds" if m == "sarimax" else "walkforward_seconds"
    wf = pd.to_numeric(t[wf_col], errors="coerce") if wf_col in t.columns else pd.Series([0.0])
    if m == "sarimax":
        costo[m] = pd.to_timedelta(t["duration"]).dt.total_seconds().sum() / 60
    else:
        costo[m] = (fit.fillna(0).sum() + wf.fillna(0).sum()) / 60
    medias[m], medianas[m] = fit.mean(), fit.median()
    wf_stats[m] = (wf.mean(), wf.median())

fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
for h, color in [(24, "#4c78a8"), (3, "#f58518")]:
    for m in MODELOS:
        rmse_t = ganadores[(m, h)]["test_metrics"]["rmse"]
        axes[0].scatter(costo[m], rmse_t, s=120, color=color, edgecolor="black", zorder=3)
        axes[0].annotate(f"{NOMBRE[m]} {h} h", (costo[m], rmse_t),
                         textcoords="offset points", xytext=(8, 4), fontsize=8)
axes[0].set_xscale("log")
axes[0].set_xlabel("Costo total de la búsqueda (minutos, escala log)")
axes[0].set_ylabel("RMSE de test (MW)")
axes[0].set_title("Costo computacional vs error de test")

comp = pd.DataFrame({"media": pd.Series(medias), "mediana": pd.Series(medianas)})
comp.plot.bar(ax=axes[1], rot=0, color=["#4c78a8", "#f58518"])
for cont in axes[1].containers:
    axes[1].bar_label(cont, fmt="%.2f", fontsize=8, padding=2)
axes[1].set_yscale("log")
axes[1].set_ylabel("Segundos por ajuste (escala log)")
axes[1].set_xlabel("")
axes[1].set_title("Tiempo de ajuste por trial: media vs mediana")
fig.suptitle("Costo computacional de la búsqueda", y=1.02)
fig.tight_layout()
guardar(fig, "integral", "costo-vs-error",
        afirmacion="Costos totales (1,1 / 7,1 / 132 / 387 min) y tiempo de ajuste por trial (línea de costos).")

print("costo total (min): " + ", ".join(f"{NOMBRE[m]}={costo[m]:.1f}" for m in MODELOS))
print("ajuste por trial (s): " +
      ", ".join(f"{NOMBRE[m]}: media={medias[m]:.2f}, mediana={medianas[m]:.2f}" for m in MODELOS))
print("walk-forward por trial (s): " +
      ", ".join(f"{NOMBRE[m]}: media={wf_stats[m][0]:.2f}, mediana={wf_stats[m][1]:.2f}" for m in MODELOS))

## 7. Manifiesto y cifras clave

In [ ]:
# Manifiesto archivo -> afirmacion del informe (se guarda en output/README.md)
tabla = "| Archivo | Grupo | Horizonte | Afirma |\n|---|---|---|---|\n" + "\n".join(
    f"| {r['archivo']} | {r['grupo']} | {r['horizonte'] if r['horizonte'] is not None else '---'} | {r['afirmacion']} |"
    for r in MANIFIESTO)
readme = f"""# Figuras del informe (generadas por `[stage-fina]figuras-informe.ipynb`)

Cada figura se regenera ejecutando el notebook desde `notebooks/`. Residuo = observado - predicho.

{tabla}
"""
(OUT / "README.md").write_text(readme, encoding="utf-8")
print(tabla[:400])
print(f"\n{len(MANIFIESTO)} figuras; manifiesto escrito en {OUT / 'README.md'}")

In [ ]:
# Cifras clave del informe, calculadas en el momento (para contrastar el texto)
filas = []
for h in (24, 3):
    for m in MODELOS:
        y = resid[(m, h)]["sesgos_sistematicos"]
        ae = (pred[(m, h)]["predicted"] - pred[(m, h)]["actual"]).abs()
        filas.append({
            "horizonte": f"{h} h", "modelo": NOMBRE[m],
            "RMSE test": f"{ganadores[(m, h)]['test_metrics']['rmse']:.2f}",
            "p90 |error|": f"{ae.quantile(0.9):.2f}",
            "MAE invierno": f"{y['mae_by_season'].get('invierno', np.nan):.2f}",
            "MAE verano": f"{y['mae_by_season'].get('verano', np.nan):.2f}",
            "ratio v/i": f"{y['mae_by_season'].get('verano', np.nan) / y['mae_by_season'].get('invierno', np.nan):.2f}",
            "sesgo medio": f"{y['mean_bias']:+.2f}",
        })
resumen = pd.DataFrame(filas)
print(resumen.to_string(index=False))